# NatGasMoE Classification + Positioning — Optuna Optimization

Regime-Aware Heterogeneous MoE (TCN + MDN) with:
- **4-class quartile classification** (<25th / <50th / >50th / >75th percentile return)
- **tanh positioning head** mapping class probabilities to exposure in [-1, +1]
- **Weather features** (HDD/CDD degree days + spline HDD basis)

Pipeline:
1. Load NG futures OHLCV + EIA storage + weather
2. Build datasets with `NGMoEDataBuilder` (technicals + weather + regime + 4-class target)
3. Optuna search over MoE architecture + positioning loss weights
4. Train final model, diagnostics, backtest

In [ ]:
from dotenv import load_dotenv; load_dotenv('/workspace/MacrOS-Intel/MacrOSINT/dot.env')

import os
import sys
import copy
import math
import time as _time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

try:
    import optuna
    from optuna.pruners import MedianPruner
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA = True
except ImportError:
    print('optuna not installed -- pip install optuna')
    _OPTUNA = False

warnings.filterwarnings('ignore')

print(f'PyTorch : {torch.__version__}')
print(f'Optuna  : {optuna.__version__ if _OPTUNA else "N/A"}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.ng_moe import (
    MoEConfig,
    NatGasMoE,
    MoELoss,
)
from CTAFlow.models.deep_learning.multi_branch.ng_moe_dataset import (
    NGMoEDataConfig,
    NGMoEDataBuilder,
    NGMoEWindowDataset,
    REGIME_COLS,
    build_datasets,
)

# macrOS-Int for EIA + weather
sys.path.insert(0, r'C:\Users\nicho\PycharmProjects\macrOS-Int')
from MacrOSINT.data.sources.eia.api_tools import NatGasHelper
from MacrOSINT.models.energy.natgas_storage_forecast import (
    NatGasStorageForecaster,
    fetch_storage_data,
    ConsensusForecast,
)

# Device
def _select_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        t = torch.zeros(1, device='cuda')
        _ = t + 1
        return 'cuda'
    except RuntimeError as e:
        print(f'CUDA unusable ({e}) -- CPU fallback')
        return 'cpu'

DEVICE = _select_device()
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
# --- Paths (adjust for Colab / RunPod / local) ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    SAVE_DIR    = Path('/content/drive/MyDrive/results/ng_moe_cls')
    EIA_HDF     = '/content/drive/MyDrive/model_data/ng_eia_cache.hdf'
    WEATHER_HDF = '/content/drive/MyDrive/model_data/weather.hdf'
else:
    SAVE_DIR    = Path('/workspace/results/ng_moe_cls')
    EIA_HDF     = '/workspace/model_data/ng_eia_cache.hdf'
    WEATHER_HDF = '/workspace/model_data/weather.hdf'

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# --- Classification config ---
N_CLASSES = 4           # quartile-based: <25 / <50 / >50 / >75
USE_POSITIONING = True  # tanh exposure head
USE_VSN = True          # grouped variable selection (weather + technicals)
TARGET_HORIZON = 1      # 1-day forward return

# --- Optuna ---
N_TRIALS       = 40
MAX_EPOCHS_OPT = 80
OPT_PATIENCE   = 10

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Save dir    : {SAVE_DIR}')
print(f'EIA HDF     : {EIA_HDF}')
print(f'Weather HDF : {WEATHER_HDF}')
print(f'Classes     : {N_CLASSES}')
print(f'Positioning : {USE_POSITIONING}')
print(f'VSN         : {USE_VSN}')

## 2. Load Price Data

In [ ]:
import yfinance as yf

price_df = yf.download('NG=F', start='2010-01-01')
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.get_level_values(0)
price_df.columns = [c.lower() for c in price_df.columns]
price_df = price_df.dropna(subset=['close'])

print(f'Price data : {price_df.shape}')
print(f'Date range : {price_df.index[0].date()} to {price_df.index[-1].date()}')
price_df.tail(3)

## 3. Load EIA Storage + Consensus

In [ ]:
START = price_df.index[0].strftime('%Y-%m')
END   = price_df.index[-1].strftime('%Y-%m')

eia_cache    = NatGasStorageForecaster.load_eia_cache(hdf_path=EIA_HDF)
storage_wkly = eia_cache.get('storage')

if storage_wkly is not None and not storage_wkly.empty:
    print(f'EIA storage from cache : {storage_wkly.shape}')
else:
    ng_helper    = NatGasHelper()
    storage_wkly = fetch_storage_data(ng_helper, start=START, end=END)
    NatGasStorageForecaster.save_eia_cache(storage=storage_wkly, hdf_path=EIA_HDF)
    print(f'EIA storage from API   : {storage_wkly.shape}')

# Consensus forecast for surprise features
try:
    cf = ConsensusForecast()
    cf.fit(storage_wkly['storage_change'])
    surprise_df = cf.transform()
    storage_wkly = storage_wkly.join(
        surprise_df[['consensus_est', 'surprise']], how='left'
    )
    print(f'ConsensusForecast added')
except Exception as e:
    print(f'ConsensusForecast failed ({e}) -- using rolling 4-week proxy')
    chg = storage_wkly['storage_change']
    storage_wkly['consensus_est'] = chg.rolling(4).mean()
    storage_wkly['surprise']      = chg - storage_wkly['consensus_est']

print(f'Weekly columns: {storage_wkly.columns.tolist()}')
storage_wkly.tail(3)

## 4. Load Population-Weighted Weather

HDD/CDD degree-day features with spline HDD basis for non-linear cold demand response.

In [ ]:
daily_weather = NatGasStorageForecaster.load_weather_hdf(hdf_path=WEATHER_HDF)

if daily_weather is not None and not daily_weather.empty:
    daily_weather = daily_weather[
        (daily_weather.index >= price_df.index[0]) &
        (daily_weather.index <= price_df.index[-1])
    ]
    print(f'Weather from cache : {daily_weather.shape}')
    print(f'Date range : {daily_weather.index[0].date()} to {daily_weather.index[-1].date()}')
    print(f'Columns    : {daily_weather.columns.tolist()}')
else:
    from MacrOSINT.models.weather.population_weather import PopulationWeatherGrid
    print('No weather cache -- fetching via PopulationWeatherGrid...')
    forecaster = NatGasStorageForecaster()
    forecaster.setup()
    daily_weather = forecaster._fetch_weather_by_epoch(
        price_df.index[0].date(), price_df.index[-1].date()
    )
    NatGasStorageForecaster.save_weather_hdf(daily_weather, hdf_path=WEATHER_HDF)
    print(f'Weather fetched and cached: {daily_weather.shape}')

daily_weather.tail(3)

## 5. Build Datasets

4-class quartile classification on expanding quantile boundaries (causal, no lookahead).
Weather features (HDD, CDD, spline basis, momentum) are included via `daily_weather`.

In [ ]:
data_cfg = NGMoEDataConfig(
    seq_len=20,
    ae_window=21,
    target='price_return',
    target_horizon=TARGET_HORIZON,
    n_classes=N_CLASSES,
)

train_ds, val_ds, test_ds, meta = build_datasets(
    price_df, storage_wkly,
    daily_weather=daily_weather,
    config=data_cfg,
    train_frac=0.70,
    val_frac=0.15,
    monday_only=False,
)

n_features = meta['n_features']
feature_groups = meta['feature_groups']
# Build {group_name: n_features} for VSN
FEATURE_GROUP_SIZES = {k: len(v) for k, v in feature_groups.items()}

print(f'Target       : {data_cfg.target} (horizon={data_cfg.target_horizon}d)')
print(f'Classes      : {N_CLASSES} (quartile distribution)')
print(f'Features     : {n_features}')
print(f'Regime cols  : {len(meta["regime_cols"])}')
print(f'Train samples: {len(train_ds)}')
print(f'Val samples  : {len(val_ds)}')
print(f'Test samples : {len(test_ds)}')

for split_name, (start, end, count) in meta['splits'].items():
    print(f'  {split_name:5s}: {start.date()} - {end.date()}  ({count} days)')

# Feature group summary
print(f'\nFeature groups ({len(FEATURE_GROUP_SIZES)}):')
for gname, gsize in FEATURE_GROUP_SIZES.items():
    cols = feature_groups[gname]
    print(f'  {gname:15s}: {gsize:2d} features  [{cols[0]}, ..., {cols[-1]}]')
assert sum(FEATURE_GROUP_SIZES.values()) == n_features, \
    f'Group sizes {sum(FEATURE_GROUP_SIZES.values())} != n_features {n_features}'

# Verify classification target
sample = train_ds[0]
print(f'\nSample length: {len(sample)} elements')
if len(sample) == 5:
    x_seq, ae_input, y_ret, y_std, y_class = sample
    print(f'x_seq={tuple(x_seq.shape)}, ae_input={tuple(ae_input.shape)}, '
          f'y_ret={y_ret.item():.4f}, y_std={y_std.item():.4f}, y_class={y_class.item()}')
else:
    x_seq, ae_input, y_ret, y_std = sample
    print(f'x_seq={tuple(x_seq.shape)}, ae_input={tuple(ae_input.shape)}')

# Class distribution
if train_ds.has_classes:
    train_classes = train_ds.y_class[train_ds.indices]
    for c in range(N_CLASSES):
        pct = (train_classes == c).mean() * 100
        print(f'  Class {c}: {pct:.1f}%')

## 6. Selection Score

Composite metric balancing classification accuracy, Sharpe, and positioning PnL.

In [ ]:
def compute_val_metrics(model, val_loader, loss_fn, device):
    """Evaluate model on validation set, return comprehensive metrics dict."""
    model.eval()
    all_losses = {k: [] for k in ['total_loss', 'return_loss', 'vol_loss',
                                    'ce_loss', 'positioning_loss', 'vsn_entropy_loss']}
    all_positions = []
    all_returns = []
    all_pred_returns = []
    all_pred_classes = []
    all_true_classes = []
    all_vsn_weights = []
    gate_vals = []
    n_total = 0

    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(device)
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            x_seq = x_seq.to(device)
            ae_in = ae_in.to(device)
            y_ret = y_ret.to(device)
            y_std = y_std.to(device)

            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)

            for k in all_losses:
                if k in losses:
                    v = losses[k]
                    all_losses[k].append((v.item() if torch.is_tensor(v) else v) * len(y_ret))

            n_total += len(y_ret)
            all_pred_returns.append(out['pred_return'].cpu().numpy())
            all_returns.append(y_ret.cpu().numpy())
            gate_vals.append(out['shared_gate'].item())

            if 'position' in out:
                all_positions.append(out['position'].cpu().numpy())

            if 'class_logits' in out and y_cls is not None:
                all_pred_classes.append(out['class_logits'].argmax(dim=-1).cpu().numpy())
                all_true_classes.append(y_cls.cpu().numpy())

            if 'vsn_weights' in out:
                all_vsn_weights.append(out['vsn_weights'].cpu().numpy())

    # --- Core losses ---
    metrics = {}
    for k, vals in all_losses.items():
        if vals:
            metrics[k] = sum(vals) / max(n_total, 1)
    metrics['val_loss'] = metrics.get('total_loss', 0)
    metrics['shared_gate'] = np.mean(gate_vals) if gate_vals else 0

    # --- Return prediction quality ---
    pred_ret = np.concatenate(all_pred_returns)
    actual_ret = np.concatenate(all_returns)
    metrics['return_mae'] = np.abs(pred_ret - actual_ret).mean()
    metrics['return_corr'] = np.corrcoef(pred_ret, actual_ret)[0, 1] if len(pred_ret) > 2 else 0
    dir_mask = np.abs(actual_ret) > 1e-6
    metrics['direction_accuracy'] = (np.sign(pred_ret[dir_mask]) == np.sign(actual_ret[dir_mask])).mean() if dir_mask.any() else 0

    # --- Classification metrics ---
    if all_pred_classes and all_true_classes:
        pc = np.concatenate(all_pred_classes)
        ac = np.concatenate(all_true_classes)
        metrics['accuracy'] = (pc == ac).mean()
        n_cls = int(ac.max()) + 1
        for c in range(n_cls):
            mask = ac == c
            if mask.sum() > 0:
                metrics[f'acc_class_{c}'] = (pc[mask] == c).mean()
                metrics[f'n_class_{c}'] = int(mask.sum())

    # --- Positioning metrics ---
    if all_positions:
        positions = np.concatenate(all_positions)
        strategy_ret = positions * actual_ret
        cum_ret = np.cumsum(strategy_ret)
        metrics['mean_strategy_ret'] = strategy_ret.mean()
        metrics['total_strategy_ret'] = cum_ret[-1] if len(cum_ret) > 0 else 0
        std = strategy_ret.std()
        metrics['sharpe'] = (strategy_ret.mean() / std * np.sqrt(252)) if std > 0 else 0.0

        downside = strategy_ret[strategy_ret < 0]
        down_std = downside.std() if len(downside) > 1 else 1e-6
        metrics['sortino'] = (strategy_ret.mean() / down_std * np.sqrt(252)) if down_std > 0 else 0.0

        gains = strategy_ret[strategy_ret > 0].sum()
        losses_sum = abs(strategy_ret[strategy_ret < 0].sum())
        metrics['profit_factor'] = gains / max(losses_sum, 1e-8)
        metrics['win_rate'] = (strategy_ret > 0).mean()
        metrics['mean_abs_position'] = np.abs(positions).mean()
        metrics['position_std'] = positions.std()

        # Max drawdown
        running_max = np.maximum.accumulate(cum_ret)
        drawdown = cum_ret - running_max
        metrics['max_drawdown'] = drawdown.min()

        # Long/short split
        long_mask = positions > 0.01
        short_mask = positions < -0.01
        if long_mask.any():
            metrics['long_win_rate'] = (strategy_ret[long_mask] > 0).mean()
            metrics['long_pct'] = long_mask.mean()
        if short_mask.any():
            metrics['short_win_rate'] = (strategy_ret[short_mask] > 0).mean()
            metrics['short_pct'] = short_mask.mean()

    # --- VSN group weights ---
    if all_vsn_weights:
        vsn_w = np.concatenate(all_vsn_weights)
        group_names = list(FEATURE_GROUP_SIZES.keys())
        for i, name in enumerate(group_names):
            metrics[f'vsn_{name}'] = vsn_w[:, i].mean()

    return metrics


def selection_score(metrics: dict) -> float:
    """Composite score for Optuna maximization."""
    if USE_POSITIONING:
        sharpe = metrics.get('sharpe', 0.0)
        sortino = metrics.get('sortino', 0.0)
        pf = metrics.get('profit_factor', 1e-8)
        acc = metrics.get('accuracy', 0.25)
        return (
            0.25 * sharpe
            + 0.30 * sortino
            + 0.20 * math.log(max(pf, 1e-8))
            + 0.25 * (acc - 0.25) * 10  # above chance
        )
    else:
        acc = metrics.get('accuracy', 0.25)
        return -metrics['val_loss'] + 5.0 * (acc - 0.25)


def _fmt_metrics(metrics: dict, prefix: str = '') -> str:
    """Format metrics dict into a compact multi-line summary."""
    lines = []
    # Losses
    loss_keys = ['val_loss', 'return_loss', 'vol_loss', 'ce_loss', 'positioning_loss']
    loss_parts = [f'{k}={metrics[k]:.5f}' for k in loss_keys if k in metrics]
    if loss_parts:
        lines.append(f'{prefix}Losses   : {", ".join(loss_parts)}')

    # Classification
    if 'accuracy' in metrics:
        cls_parts = [f'acc={metrics["accuracy"]:.3f}']
        for c in range(10):
            k = f'acc_class_{c}'
            if k in metrics:
                cls_parts.append(f'c{c}={metrics[k]:.3f}({metrics.get(f"n_class_{c}", 0)})')
        lines.append(f'{prefix}Class    : {", ".join(cls_parts)}')

    # Positioning
    if 'sharpe' in metrics:
        pos_parts = [
            f'sharpe={metrics["sharpe"]:.3f}',
            f'sortino={metrics["sortino"]:.3f}',
            f'pf={metrics.get("profit_factor", 0):.2f}',
            f'win={metrics.get("win_rate", 0):.1%}',
            f'maxDD={metrics.get("max_drawdown", 0):.4f}',
        ]
        lines.append(f'{prefix}Position : {", ".join(pos_parts)}')
        pos_detail = [
            f'|pos|={metrics.get("mean_abs_position", 0):.3f}',
            f'pos_std={metrics.get("position_std", 0):.3f}',
        ]
        if 'long_pct' in metrics:
            pos_detail.append(f'long={metrics["long_pct"]:.1%}(win={metrics.get("long_win_rate", 0):.1%})')
        if 'short_pct' in metrics:
            pos_detail.append(f'short={metrics["short_pct"]:.1%}(win={metrics.get("short_win_rate", 0):.1%})')
        lines.append(f'{prefix}         : {", ".join(pos_detail)}')

    # Return prediction
    ret_parts = [
        f'mae={metrics.get("return_mae", 0):.5f}',
        f'corr={metrics.get("return_corr", 0):.3f}',
        f'dir_acc={metrics.get("direction_accuracy", 0):.3f}',
    ]
    lines.append(f'{prefix}Returns  : {", ".join(ret_parts)}')

    # VSN
    vsn_parts = [f'{k.replace("vsn_", "")}={v:.3f}' for k, v in metrics.items() if k.startswith('vsn_')]
    if vsn_parts:
        lines.append(f'{prefix}VSN      : {", ".join(vsn_parts)}')

    return '\n'.join(lines)


print('Metrics + selection score defined')

## 7. Optuna Objective

In [ ]:
_trial_log = []


def _make_loaders(train_ds, val_ds, bs):
    tr = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True)
    va = DataLoader(val_ds, batch_size=bs, shuffle=False, drop_last=False)
    return tr, va


def objective(trial):
    t0 = _time.time()

    # --- Architecture ---
    d_latent       = trial.suggest_categorical('d_latent', [16, 32, 64])
    d_ae_hidden    = trial.suggest_categorical('d_ae_hidden', [64, 128, 256])
    n_tcn_experts  = trial.suggest_int('n_tcn_experts', 2, 5)
    n_mdn_experts  = trial.suggest_int('n_mdn_experts', 1, 4)
    top_k          = trial.suggest_int('top_k', 2, min(n_tcn_experts + n_mdn_experts, 5))
    tcn_width      = trial.suggest_categorical('tcn_width', [32, 64, 128])
    tcn_depth      = trial.suggest_int('tcn_depth', 2, 4)
    mdn_hidden     = trial.suggest_categorical('mdn_hidden', [32, 64, 128])
    mdn_n_comp     = trial.suggest_int('mdn_n_components', 2, 6)
    shared_dim     = trial.suggest_categorical('shared_expert_dim', [32, 64, 128])
    dropout        = trial.suggest_float('dropout', 0.05, 0.4)
    pos_hidden     = trial.suggest_categorical('positioning_hidden_dim', [16, 32, 64])

    # --- VSN hyperparameters ---
    if USE_VSN:
        vsn_d_model      = trial.suggest_categorical('vsn_d_model', [16, 32, 64])
        vsn_temperature   = trial.suggest_float('vsn_temperature', 0.5, 3.0)
        vsn_entropy_weight = trial.suggest_float('vsn_entropy_weight', 0.01, 0.3, log=True)
        vsn_min_weight     = trial.suggest_float('vsn_min_weight', 0.0, 0.1)
    else:
        vsn_d_model, vsn_temperature, vsn_entropy_weight, vsn_min_weight = 32, 1.5, 0.1, 0.05

    # --- Loss weights ---
    ce_weight      = trial.suggest_float('ce_weight', 0.3, 3.0, log=True)
    pnl_weight     = trial.suggest_float('positioning_pnl_weight', 0.1, 2.0, log=True)
    nll_weight     = trial.suggest_float('nll_weight', 0.01, 0.5, log=True)
    kl_weight      = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)
    recon_weight   = trial.suggest_float('recon_weight', 0.01, 0.5, log=True)
    balance_w      = trial.suggest_float('load_balance_weight', 0.001, 0.1, log=True)
    entropy_w      = trial.suggest_float('entropy_reg_weight', 0.001, 0.1, log=True)
    tc_cost        = trial.suggest_float('tc_cost', 0.0, 0.001)

    # --- Training ---
    lr             = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    weight_decay   = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
    batch_size     = trial.suggest_categorical('batch_size', [16, 32, 64])

    config = MoEConfig(
        n_features=n_features,
        seq_len=data_cfg.seq_len,
        f_ae=12,
        ae_window=data_cfg.ae_window,
        d_latent=d_latent,
        d_ae_hidden=d_ae_hidden,
        kl_weight=kl_weight,
        recon_weight=recon_weight,
        n_tcn_experts=n_tcn_experts,
        n_mdn_experts=n_mdn_experts,
        top_k=top_k,
        shared_expert_dim=shared_dim,
        tcn_channels=[tcn_width] * tcn_depth,
        mdn_hidden_dims=[mdn_hidden, mdn_hidden // 2],
        mdn_n_components=mdn_n_comp,
        dropout=dropout,
        load_balance_weight=balance_w,
        entropy_reg_weight=entropy_w,
        nll_weight=nll_weight,
        n_classes=N_CLASSES,
        use_positioning_head=USE_POSITIONING,
        positioning_hidden_dim=pos_hidden,
        ce_weight=ce_weight,
        positioning_pnl_weight=pnl_weight,
        tc_cost=tc_cost,
        use_vsn=USE_VSN,
        vsn_d_model=vsn_d_model,
        vsn_temperature=vsn_temperature,
        vsn_entropy_weight=vsn_entropy_weight,
        vsn_min_weight=vsn_min_weight,
    )

    model = NatGasMoE(
        config,
        feature_group_sizes=FEATURE_GROUP_SIZES if USE_VSN else None,
    ).to(DEVICE)
    loss_fn = MoELoss(config)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    train_loader, val_loader = _make_loaders(train_ds, val_ds, batch_size)

    n_params = sum(p.numel() for p in model.parameters())
    best_score = float('-inf')
    best_metrics = {}
    patience_cnt = 0

    for epoch in range(1, MAX_EPOCHS_OPT + 1):
        # --- Train ---
        model.train()
        for batch in train_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(DEVICE)
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            x_seq = x_seq.to(DEVICE)
            ae_in = ae_in.to(DEVICE)
            y_ret = y_ret.to(DEVICE)
            y_std = y_std.to(DEVICE)

            optimizer.zero_grad()
            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            losses['total_loss'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # --- Validate ---
        metrics = compute_val_metrics(model, val_loader, loss_fn, DEVICE)
        score = selection_score(metrics)

        trial.report(score, epoch)
        if trial.should_prune():
            elapsed = _time.time() - t0
            print(f'  Trial {trial.number:3d} PRUNED  epoch={epoch}  '
                  f'score={score:.3f}  val_loss={metrics["val_loss"]:.5f}  '
                  f'({elapsed:.0f}s)')
            _trial_log.append({
                'trial': trial.number, 'status': 'PRUNED',
                'epoch': epoch, 'score': score,
                'val_loss': metrics['val_loss'],
                'time': elapsed,
            })
            raise optuna.TrialPruned()

        if score > best_score:
            best_score = score
            best_metrics = metrics.copy()
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= OPT_PATIENCE:
                break

    elapsed = _time.time() - t0

    # --- Verbose trial summary ---
    print(f'\n{"="*70}')
    print(f'Trial {trial.number:3d} COMPLETE  epochs={epoch}  '
          f'score={best_score:.3f}  params={n_params:,}  ({elapsed:.0f}s)')
    print(f'  Arch: tcn={n_tcn_experts} mdn={n_mdn_experts} top_k={top_k} '
          f'd_latent={d_latent} tcn=[{tcn_width}]*{tcn_depth} '
          f'vsn_d={vsn_d_model if USE_VSN else "off"}')
    print(_fmt_metrics(best_metrics, prefix='  '))
    print(f'{"="*70}')

    log_entry = {
        'trial': trial.number, 'status': 'COMPLETE',
        'epoch': epoch, 'score': best_score,
        'val_loss': best_metrics.get('val_loss', 0),
        'accuracy': best_metrics.get('accuracy', 0),
        'sharpe': best_metrics.get('sharpe', 0),
        'sortino': best_metrics.get('sortino', 0),
        'profit_factor': best_metrics.get('profit_factor', 0),
        'win_rate': best_metrics.get('win_rate', 0),
        'max_drawdown': best_metrics.get('max_drawdown', 0),
        'direction_accuracy': best_metrics.get('direction_accuracy', 0),
        'return_corr': best_metrics.get('return_corr', 0),
        'mean_abs_position': best_metrics.get('mean_abs_position', 0),
        'n_params': n_params,
        'time': elapsed,
    }
    # Add per-class accuracy
    for c in range(N_CLASSES):
        log_entry[f'acc_class_{c}'] = best_metrics.get(f'acc_class_{c}', 0)
    # Add VSN weights
    for name in FEATURE_GROUP_SIZES:
        log_entry[f'vsn_{name}'] = best_metrics.get(f'vsn_{name}', 0)
    _trial_log.append(log_entry)

    return best_score

print(f'Objective defined (n_features={n_features}, groups={list(FEATURE_GROUP_SIZES.keys())})')

## 8. Run Optuna Optimization

In [ ]:
pos_tag = '_pos' if USE_POSITIONING else ''
STUDY_NAME = f'ng_moe_cls{N_CLASSES}{pos_tag}'

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f'Study   : {STUDY_NAME}')
print(f'Trials  : {N_TRIALS}')
print(f'Classes : {N_CLASSES}')
print(f'Head    : {"tanh positioning" if USE_POSITIONING else "classification only"}')
print('-' * 60)

In [ ]:
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest trial: #{study.best_trial.number}')
print(f'Best score: {study.best_value:.4f}')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Save study artifacts
import json

best_params = dict(study.best_params)
best_params['best_value'] = study.best_value
best_params['n_classes'] = N_CLASSES
best_params['use_positioning'] = USE_POSITIONING
best_params['n_features'] = n_features
best_params['target_horizon'] = TARGET_HORIZON

with open(SAVE_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

trial_df = pd.DataFrame(_trial_log)
trial_df.to_csv(SAVE_DIR / 'trial_log.csv', index=False)
print(f'Saved to {SAVE_DIR}')
trial_df.sort_values('score', ascending=False).head(10)

## 9. Optuna Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Optimization history
comp = trial_df[trial_df['status'] == 'COMPLETE'].copy()
best_so_far = comp['score'].expanding().max()
axes[0, 0].scatter(comp['trial'], comp['score'], s=20, alpha=0.6, label='trial score')
axes[0, 0].plot(comp['trial'].values, best_so_far.values, color='red', lw=1.5, label='best so far')
axes[0, 0].set_xlabel('Trial'); axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Optimization History'); axes[0, 0].legend(fontsize=8)

# 2. Param importances
try:
    imp = optuna.importance.get_param_importances(study)
    names = list(imp.keys())[:12]
    vals  = list(imp.values())[:12]
    axes[0, 1].barh(names, vals, color='steelblue', edgecolor='black')
    axes[0, 1].set_xlabel('Importance')
    axes[0, 1].set_title('Top Hyperparameter Importance')
except Exception:
    axes[0, 1].text(0.5, 0.5, 'Not enough data', ha='center', va='center',
                    transform=axes[0, 1].transAxes)

# 3. Score components (accuracy, sharpe, sortino)
if 'accuracy' in comp.columns:
    axes[1, 0].scatter(comp['accuracy'], comp['score'], s=20, alpha=0.6, c='green')
    axes[1, 0].set_xlabel('Val Accuracy'); axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_title('Score vs Accuracy')

if 'sharpe' in comp.columns:
    axes[1, 1].scatter(comp['sharpe'], comp['sortino'], s=20, alpha=0.6,
                       c=comp['score'], cmap='viridis')
    axes[1, 1].set_xlabel('Sharpe'); axes[1, 1].set_ylabel('Sortino')
    axes[1, 1].set_title('Sharpe vs Sortino (color=score)')
    plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1], label='Score')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'optuna_viz.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Train Final Model with Best Parameters

In [ ]:
bp = study.best_params
print('Best parameters:')
for k, v in sorted(bp.items()):
    print(f'  {k}: {v}')

final_cfg = MoEConfig(
    n_features=n_features,
    seq_len=data_cfg.seq_len,
    f_ae=12,
    ae_window=data_cfg.ae_window,
    d_latent=bp['d_latent'],
    d_ae_hidden=bp['d_ae_hidden'],
    kl_weight=bp['kl_weight'],
    recon_weight=bp['recon_weight'],
    n_tcn_experts=bp['n_tcn_experts'],
    n_mdn_experts=bp['n_mdn_experts'],
    top_k=bp['top_k'],
    shared_expert_dim=bp['shared_expert_dim'],
    tcn_channels=[bp['tcn_width']] * bp['tcn_depth'],
    mdn_hidden_dims=[bp['mdn_hidden'], bp['mdn_hidden'] // 2],
    mdn_n_components=bp['mdn_n_components'],
    dropout=bp['dropout'],
    load_balance_weight=bp['load_balance_weight'],
    entropy_reg_weight=bp['entropy_reg_weight'],
    nll_weight=bp['nll_weight'],
    n_classes=N_CLASSES,
    use_positioning_head=USE_POSITIONING,
    positioning_hidden_dim=bp['positioning_hidden_dim'],
    ce_weight=bp['ce_weight'],
    positioning_pnl_weight=bp['positioning_pnl_weight'],
    tc_cost=bp['tc_cost'],
    use_vsn=USE_VSN,
    vsn_d_model=bp.get('vsn_d_model', 32),
    vsn_temperature=bp.get('vsn_temperature', 1.5),
    vsn_entropy_weight=bp.get('vsn_entropy_weight', 0.1),
    vsn_min_weight=bp.get('vsn_min_weight', 0.05),
)

model = NatGasMoE(
    final_cfg,
    feature_group_sizes=FEATURE_GROUP_SIZES if USE_VSN else None,
).to(DEVICE)
loss_fn = MoELoss(final_cfg)
optimizer = optim.AdamW(model.parameters(), lr=bp['lr'], weight_decay=bp['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)

FINAL_BS = bp['batch_size']
train_loader = DataLoader(train_ds, batch_size=FINAL_BS, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=FINAL_BS, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=FINAL_BS, shuffle=False, drop_last=False)

print(f'\nModel params : {sum(p.numel() for p in model.parameters()):,}')
print(f'VSN groups   : {list(FEATURE_GROUP_SIZES.keys()) if USE_VSN else "disabled"}')
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

In [ ]:
MAX_EPOCHS = 200
PATIENCE   = 20
LOG_EVERY  = 20

loss_keys = ['total_loss', 'return_loss', 'vol_loss', 'mdn_nll_loss',
             'balance_loss', 'entropy_loss', 'ae_recon_loss', 'ae_kl_loss',
             'vsn_entropy_loss']
if N_CLASSES > 0:
    loss_keys.extend(['ce_loss', 'class_accuracy'])
if USE_POSITIONING:
    loss_keys.extend(['positioning_loss', 'mean_position', 'mean_strategy_ret'])

history = {f'train_{k}': [] for k in loss_keys}
history.update({f'val_{k}': [] for k in loss_keys})
history['shared_gate'] = []
history['val_sharpe'] = []
history['val_sortino'] = []
history['val_score'] = []
history['val_profit_factor'] = []
history['val_win_rate'] = []
history['val_max_drawdown'] = []
history['val_direction_accuracy'] = []
history['val_return_corr'] = []
# Per-group VSN weights
for gname in FEATURE_GROUP_SIZES:
    history[f'val_vsn_{gname}'] = []

best_val_loss = float('inf')
best_state    = None
best_metrics  = {}
patience_cnt  = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_train = {k: [] for k in loss_keys}
    for batch in train_loader:
        if len(batch) == 5:
            x_seq, ae_in, y_ret, y_std, y_cls = batch
            y_cls = y_cls.to(DEVICE)
        else:
            x_seq, ae_in, y_ret, y_std = batch
            y_cls = None

        x_seq = x_seq.to(DEVICE)
        ae_in = ae_in.to(DEVICE)
        y_ret = y_ret.to(DEVICE)
        y_std = y_std.to(DEVICE)

        optimizer.zero_grad()
        out = model(x_seq, ae_in)
        losses = loss_fn(out, y_ret, y_std, y_cls)
        losses['total_loss'].backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        for k in loss_keys:
            if k in losses:
                v = losses[k]
                epoch_train[k].append(v.item() if torch.is_tensor(v) else v)
    scheduler.step()

    # --- Validate (batch-level losses for history curves) ---
    model.eval()
    epoch_val = {k: [] for k in loss_keys}
    gate_vals = []
    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(DEVICE)
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            x_seq = x_seq.to(DEVICE)
            ae_in = ae_in.to(DEVICE)
            y_ret = y_ret.to(DEVICE)
            y_std = y_std.to(DEVICE)

            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            for k in loss_keys:
                if k in losses:
                    v = losses[k]
                    epoch_val[k].append(v.item() if torch.is_tensor(v) else v)
            gate_vals.append(out['shared_gate'].item())

    # Log batch-level losses
    for k in loss_keys:
        history[f'train_{k}'].append(np.mean(epoch_train[k]) if epoch_train[k] else 0)
        history[f'val_{k}'].append(np.mean(epoch_val[k]) if epoch_val[k] else 0)
    history['shared_gate'].append(np.mean(gate_vals))

    val_loss = history['val_total_loss'][-1]

    # Early stopping check
    is_new_best = val_loss < best_val_loss
    if is_new_best:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1

    # Full val metrics on log epochs, epoch 1, or new best
    should_log = (epoch % LOG_EVERY == 0) or (epoch == 1) or is_new_best

    if should_log:
        val_metrics = compute_val_metrics(model, val_loader, loss_fn, DEVICE)
        score = selection_score(val_metrics)
        if is_new_best:
            best_metrics = val_metrics.copy()
    else:
        val_metrics = {}
        score = history['val_score'][-1] if history['val_score'] else 0.0

    # Append full metrics to history (carry forward when not computed)
    def _get(key, default_key=None):
        if key in val_metrics:
            return val_metrics[key]
        hist = history.get(default_key or f'val_{key}', [])
        return hist[-1] if hist else 0

    history['val_sharpe'].append(_get('sharpe', 'val_sharpe'))
    history['val_sortino'].append(_get('sortino', 'val_sortino'))
    history['val_score'].append(score)
    history['val_profit_factor'].append(_get('profit_factor', 'val_profit_factor'))
    history['val_win_rate'].append(_get('win_rate', 'val_win_rate'))
    history['val_max_drawdown'].append(_get('max_drawdown', 'val_max_drawdown'))
    history['val_direction_accuracy'].append(_get('direction_accuracy', 'val_direction_accuracy'))
    history['val_return_corr'].append(_get('return_corr', 'val_return_corr'))
    for gname in FEATURE_GROUP_SIZES:
        history[f'val_vsn_{gname}'].append(_get(f'vsn_{gname}', f'val_vsn_{gname}'))

    # --- Verbose logging ---
    if should_log:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'\n--- Epoch {epoch:3d}/{MAX_EPOCHS} '
              f'{"[NEW BEST]" if is_new_best else ""} '
              f'(patience={patience_cnt}/{PATIENCE}, lr={lr_now:.2e}) ---')

        # Loss comparison
        print(f'  Train losses: total={history["train_total_loss"][-1]:.5f}  '
              f'ret={history["train_return_loss"][-1]:.5f}  '
              f'vol={history["train_vol_loss"][-1]:.5f}  '
              f'ce={history.get("train_ce_loss", [0])[-1]:.5f}  '
              f'pos={history.get("train_positioning_loss", [0])[-1]:.5f}')
        print(f'  Val losses  : total={val_loss:.5f}  '
              f'ret={history["val_return_loss"][-1]:.5f}  '
              f'vol={history["val_vol_loss"][-1]:.5f}  '
              f'ce={history.get("val_ce_loss", [0])[-1]:.5f}  '
              f'pos={history.get("val_positioning_loss", [0])[-1]:.5f}')

        # Full metrics block
        print(_fmt_metrics(val_metrics, prefix='  '))
        print(f'  Score={score:.3f}  gate={history["shared_gate"][-1]:.3f}')

    if patience_cnt >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

# --- Final summary ---
print(f'\n{"="*70}')
print(f'Training complete  epochs={epoch}  best_val_loss={best_val_loss:.5f}')
print(f'Best model metrics:')
print(_fmt_metrics(best_metrics, prefix='  '))
print(f'  Score={selection_score(best_metrics):.3f}')
print(f'{"="*70}')

In [ ]:
# Load best state
if best_state:
    model.load_state_dict(best_state)
    print('Loaded best model state')

## 11. Training Diagnostics

In [ ]:
ep = range(1, len(history['train_total_loss']) + 1)
n_cols = 4
fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 12))

# Row 1: Core losses
axes[0, 0].plot(ep, history['train_total_loss'], label='Train')
axes[0, 0].plot(ep, history['val_total_loss'], label='Val')
axes[0, 0].set_title('Total Loss'); axes[0, 0].legend()

axes[0, 1].plot(ep, history['train_return_loss'], label='Train')
axes[0, 1].plot(ep, history['val_return_loss'], label='Val')
axes[0, 1].set_title('Return Loss (Huber)'); axes[0, 1].legend()

axes[0, 2].plot(ep, history['train_vol_loss'], label='Train')
axes[0, 2].plot(ep, history['val_vol_loss'], label='Val')
axes[0, 2].set_title('Vol Loss'); axes[0, 2].legend()

if USE_POSITIONING and 'val_positioning_loss' in history:
    axes[0, 3].plot(ep, history['val_positioning_loss'], color='purple')
    axes[0, 3].set_title('Positioning Loss (Val)')
else:
    axes[0, 3].plot(ep, history['val_mdn_nll_loss'], color='purple')
    axes[0, 3].set_title('MDN NLL (Val)')

# Row 2: Classification + routing
if N_CLASSES > 0 and 'val_class_accuracy' in history:
    axes[1, 0].plot(ep, history['train_class_accuracy'], label='Train')
    axes[1, 0].plot(ep, history['val_class_accuracy'], label='Val')
    axes[1, 0].axhline(1.0 / N_CLASSES, color='red', linestyle='--', label='Chance')
    axes[1, 0].set_title(f'{N_CLASSES}-Class Accuracy'); axes[1, 0].legend()
else:
    axes[1, 0].plot(ep, history['val_mdn_nll_loss'], color='purple')
    axes[1, 0].set_title('MDN NLL (Val)')

axes[1, 1].plot(ep, history['val_balance_loss'], label='Balance')
axes[1, 1].plot(ep, history['val_entropy_loss'], label='Entropy')
axes[1, 1].set_title('Router Regularization'); axes[1, 1].legend()

axes[1, 2].plot(ep, history['shared_gate'], color='steelblue')
axes[1, 2].set_title('Shared Gate'); axes[1, 2].set_ylabel('alpha')

if USE_POSITIONING and 'val_mean_strategy_ret' in history:
    axes[1, 3].plot(ep, np.cumsum(history['val_mean_strategy_ret']), color='green')
    axes[1, 3].set_title('Cumulative Val Strategy Return')
    axes[1, 3].axhline(0, color='black', linewidth=0.5)
else:
    axes[1, 3].axis('off')

# Row 3: Sharpe, Sortino, Score
axes[2, 0].plot(ep, history['val_sharpe'], color='teal', label='Sharpe')
axes[2, 0].axhline(0, color='black', linewidth=0.5)
axes[2, 0].set_title('Val Sharpe'); axes[2, 0].set_xlabel('Epoch'); axes[2, 0].legend()

axes[2, 1].plot(ep, history['val_sortino'], color='darkorange', label='Sortino')
axes[2, 1].axhline(0, color='black', linewidth=0.5)
axes[2, 1].set_title('Val Sortino'); axes[2, 1].set_xlabel('Epoch'); axes[2, 1].legend()

axes[2, 2].plot(ep, history['val_score'], color='crimson', label='Score')
axes[2, 2].set_title('Val Selection Score'); axes[2, 2].set_xlabel('Epoch'); axes[2, 2].legend()

axes[2, 3].plot(ep, history['val_ce_loss'] if 'val_ce_loss' in history else [], color='navy')
axes[2, 3].set_title('CE Loss (Val)'); axes[2, 3].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Router Utilization

In [ ]:
model.eval()
all_weights = []
all_z_regime = []

with torch.no_grad():
    for batch in val_loader:
        x_seq = batch[0].to(DEVICE)
        ae_in = batch[1].to(DEVICE)
        out = model(x_seq, ae_in)
        all_weights.append(out['router_weights'].cpu().numpy())
        all_z_regime.append(out['z_regime'].cpu().numpy())

weights_np = np.concatenate(all_weights)
z_regime_np = np.concatenate(all_z_regime)

n_tcn = final_cfg.n_tcn_experts
n_mdn = final_cfg.n_mdn_experts
expert_labels = [f'TCN_{i}' for i in range(n_tcn)] + [f'MDN_{i}' for i in range(n_mdn)]
avg_w = weights_np.mean(axis=0)

print(f'{"Expert":<8} {"Avg Weight":>10} {"Active %":>9}')
print('-' * 32)
for i, label in enumerate(expert_labels):
    active_pct = (weights_np[:, i] > 0).mean() * 100
    print(f'{label:<8} {avg_w[i]:>10.4f} {active_pct:>8.1f}%')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue'] * n_tcn + ['darkorange'] * n_mdn
ax.bar(expert_labels, avg_w, color=colors, edgecolor='black')
ax.axhline(1.0 / len(expert_labels), color='red', linestyle='--', label='uniform')
ax.set_ylabel('Average Router Weight')
ax.set_title('Expert Utilization (Val)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- VSN Group Weights Visualization ---
if USE_VSN and model.vsn is not None:
    model.eval()
    all_vsn_weights = []

    with torch.no_grad():
        for batch in val_loader:
            x_seq = batch[0].to(DEVICE)
            ae_in = batch[1].to(DEVICE)
            out = model(x_seq, ae_in)
            if 'vsn_weights' in out:
                all_vsn_weights.append(out['vsn_weights'].cpu().numpy())

    if all_vsn_weights:
        vsn_w = np.concatenate(all_vsn_weights)  # (N, n_groups)
        group_names = list(FEATURE_GROUP_SIZES.keys())
        avg_vsn_w = vsn_w.mean(axis=0)

        print(f'VSN Group Selection Weights (val, {len(vsn_w)} samples):')
        print(f'{"Group":<15} {"Avg Weight":>10} {"Std":>8} {"Min":>8} {"Max":>8}')
        print('-' * 52)
        for i, name in enumerate(group_names):
            print(f'{name:<15} {avg_vsn_w[i]:>10.4f} {vsn_w[:, i].std():>8.4f} '
                  f'{vsn_w[:, i].min():>8.4f} {vsn_w[:, i].max():>8.4f}')

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        # 1. Average group weights
        colors = plt.cm.Set2(np.linspace(0, 1, len(group_names)))
        axes[0].bar(group_names, avg_vsn_w, color=colors, edgecolor='black')
        axes[0].axhline(1.0 / len(group_names), color='red', linestyle='--', label='uniform')
        axes[0].set_ylabel('Average Weight')
        axes[0].set_title('VSN Group Selection (Val)')
        axes[0].legend(fontsize=8)
        axes[0].tick_params(axis='x', rotation=45)

        # 2. Weight distribution per group (violin)
        parts = axes[1].violinplot(
            [vsn_w[:, i] for i in range(len(group_names))],
            positions=range(len(group_names)),
            showmeans=True, showmedians=True,
        )
        axes[1].set_xticks(range(len(group_names)))
        axes[1].set_xticklabels(group_names, rotation=45)
        axes[1].set_ylabel('Weight')
        axes[1].set_title('Per-Group Weight Distribution')

        # 3. Time evolution (rolling mean of group weights)
        val_dates = [val_ds.get_date(i) for i in range(min(len(vsn_w), len(val_ds)))]
        if len(val_dates) == len(vsn_w):
            for i, name in enumerate(group_names):
                rolling = pd.Series(vsn_w[:, i], index=val_dates).rolling(30, min_periods=5).mean()
                axes[2].plot(rolling.index, rolling.values, label=name)
            axes[2].set_title('VSN Weights (30d rolling mean)')
            axes[2].legend(fontsize=7, ncol=2)
            axes[2].tick_params(axis='x', rotation=30)

        plt.tight_layout()
        plt.savefig(SAVE_DIR / 'vsn_group_weights.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('VSN disabled -- skipping group weight visualization')

## 13. Prediction Analysis + Backtest

In [ ]:
def evaluate_split(ds, loader, name):
    """Run model on a dataset split, return results dict."""
    model.eval()
    preds_ret, preds_std, actuals_ret = [], [], []
    pred_classes, actual_classes = [], []
    positions = []

    with torch.no_grad():
        for batch in loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            out = model(x_seq.to(DEVICE), ae_in.to(DEVICE))

            preds_ret.append(out['pred_return'].cpu().numpy())
            preds_std.append(out['pred_std'].cpu().numpy())
            actuals_ret.append(y_ret.numpy())

            if 'class_logits' in out:
                pred_classes.append(out['class_logits'].argmax(dim=-1).cpu().numpy())
            if y_cls is not None:
                actual_classes.append(y_cls.numpy())
            if 'position' in out:
                positions.append(out['position'].cpu().numpy())

    preds_ret = np.concatenate(preds_ret)
    actuals_ret = np.concatenate(actuals_ret)
    dates = [ds.get_date(i) for i in range(len(ds))]

    # Return prediction metrics
    mae = np.abs(preds_ret - actuals_ret).mean()
    corr = np.corrcoef(preds_ret, actuals_ret)[0, 1]
    dir_mask = np.abs(actuals_ret) > 1e-6
    dir_acc = (np.sign(preds_ret[dir_mask]) == np.sign(actuals_ret[dir_mask])).mean()

    print(f'\n=== {name} ({len(ds)} samples) ===')
    print(f'  MAE: {mae:.5f}  Corr: {corr:.4f}  Dir acc: {dir_acc:.3f}')

    # Classification metrics
    if pred_classes and actual_classes:
        pc = np.concatenate(pred_classes)
        ac = np.concatenate(actual_classes)
        cls_acc = (pc == ac).mean()
        print(f'  Class accuracy: {cls_acc:.3f} (chance={1/N_CLASSES:.3f})')
        # Per-class accuracy
        for c in range(N_CLASSES):
            mask = ac == c
            if mask.sum() > 0:
                print(f'    Class {c}: {(pc[mask] == c).mean():.3f} ({mask.sum()} samples)')

    # Positioning backtest
    if positions:
        pos = np.concatenate(positions)
        strat_ret = pos * actuals_ret
        cum_ret = np.cumsum(strat_ret)
        sharpe = strat_ret.mean() / strat_ret.std() * np.sqrt(252) if strat_ret.std() > 0 else 0
        downside = strat_ret[strat_ret < 0]
        sortino = strat_ret.mean() / downside.std() * np.sqrt(252) if len(downside) > 1 and downside.std() > 0 else 0
        max_dd = (cum_ret - np.maximum.accumulate(cum_ret)).min()
        print(f'  Positioning: Sharpe={sharpe:.3f}  Sortino={sortino:.3f}  MaxDD={max_dd:.4f}')
        print(f'  Mean |pos|={np.abs(pos).mean():.3f}  Total return={cum_ret[-1]:.4f}')
        return {'dates': dates, 'positions': pos, 'strategy_ret': strat_ret, 'cum_ret': cum_ret}

    return {'dates': dates, 'preds': preds_ret, 'actuals': actuals_ret}

val_results = evaluate_split(val_ds, val_loader, 'Validation')
test_results = evaluate_split(test_ds, test_loader, 'Test')

In [ ]:
# Backtest charts
if 'cum_ret' in test_results:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    # Cumulative return
    axes[0, 0].plot(test_results['dates'], test_results['cum_ret'], color='green')
    axes[0, 0].axhline(0, color='black', linewidth=0.5)
    axes[0, 0].set_title('Test: Cumulative Strategy Return')
    axes[0, 0].set_ylabel('Cumulative log return')

    # Position distribution
    axes[0, 1].hist(test_results['positions'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 1].set_title('Test: Position Distribution')
    axes[0, 1].set_xlabel('Position [-1, +1]')

    # Rolling Sharpe (60-day)
    sr = pd.Series(test_results['strategy_ret'])
    roll_sharpe = sr.rolling(60).mean() / sr.rolling(60).std() * np.sqrt(252)
    axes[1, 0].plot(test_results['dates'], roll_sharpe.values, color='purple')
    axes[1, 0].axhline(0, color='black', linewidth=0.5)
    axes[1, 0].set_title('Test: Rolling 60d Sharpe')

    # Val cumulative return
    if 'cum_ret' in val_results:
        axes[1, 1].plot(val_results['dates'], val_results['cum_ret'], color='blue', label='Val')
        axes[1, 1].plot(test_results['dates'], test_results['cum_ret'], color='green', label='Test')
        axes[1, 1].axhline(0, color='black', linewidth=0.5)
        axes[1, 1].set_title('Val vs Test Cumulative Return')
        axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'backtest_charts.png', dpi=150, bbox_inches='tight')
    plt.show()

## 14. Save Final Model

In [ ]:
import json
from dataclasses import asdict

model_path = SAVE_DIR / 'ng_moe_cls_best.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': asdict(final_cfg),
    'data_config': asdict(data_cfg),
    'best_params': best_params,
    'n_features': n_features,
    'feature_cols': meta['feature_cols'],
    'regime_cols': meta['regime_cols'],
}, model_path)

print(f'Model saved to {model_path}')
print(f'Config: {N_CLASSES} classes, positioning={USE_POSITIONING}')
print(f'Features: {n_features} technical + 12 regime')
weather_cols = [c for c in meta['feature_cols'] if c.startswith('dd_') or c.startswith('wtd_') or 'hdd' in c.lower()]
print(f'Weather features: {len(weather_cols)}')